In [1]:
import subprocess, sys, os

result = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                         capture_output=True, text=True)
ver = result.stdout.strip()
print(f'Current torch: {ver}')

if '+cpu' in ver or ver == '':
    os.system('pip uninstall -y torch torchvision torchaudio -q')
    os.system('pip install -q torch==2.1.2+cu118 torchvision torchaudio '
              '--index-url https://download.pytorch.org/whl/cu118')
    print('Restart runtime and re-run from Cell 0.')
else:
    print('CUDA torch already installed')

os.system('pip uninstall -y torchao -q')
os.system(
    'pip install -q bitsandbytes>=0.43.0 transformers>=4.40.0 accelerate>=0.27.0 '
    'peft>=0.10.0 "sentence-transformers>=3.0.0" '
    'rouge-score bert-score nltk sacrebleu '
    'pandas numpy scikit-learn matplotlib datasets tqdm'
)

import ast, json, zipfile, warnings, time, gc, pickle, math
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

Current torch: 2.11.0+cu128
CUDA torch already installed
PyTorch: 2.11.0+cu128 | CUDA available: True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
print(os.path.exists('/content/drive/MyDrive/semeval_august27neil_results'))

True


In [4]:
ZIP_PATH    = '/content/drive/MyDrive/semeval2025_task7_data.zip'
EXTRACT_DIR = '/content/semeval2025'
OUTPUT_DIR  = '/content/drive/MyDrive/semeval_august27neil_results'
CKPT_DIR    = os.path.join(OUTPUT_DIR, 'training_checkpoint_v3_lowlr')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(EXTRACT_DIR):
    print('Extracting raw dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Raw dataset already extracted')

def find_folder(root, target):
    for dirpath, dirnames, _ in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

TRAIN_DEV_DIR = find_folder(EXTRACT_DIR, 'train_dev_sets')
TEST_SET_DIR  = find_folder(EXTRACT_DIR, 'test_set')
print(f'TRAIN_DEV_DIR : {TRAIN_DEV_DIR}')
print(f'TEST_SET_DIR  : {TEST_SET_DIR}')

FINAL_ADAPTER_PATH = os.path.join(OUTPUT_DIR, 'lora_best_dev')
if not os.path.exists(FINAL_ADAPTER_PATH):
    FINAL_ADAPTER_PATH = os.path.join(CKPT_DIR, 'adapter')
print(f'LoRA adapter to load: {FINAL_ADAPTER_PATH}')
assert os.path.exists(FINAL_ADAPTER_PATH), 'Adapter not found — check OUTPUT_DIR/CKPT_DIR names!'

Extracting raw dataset...
TRAIN_DEV_DIR : /content/semeval2025/train_dev_sets
TEST_SET_DIR  : /content/semeval2025/test_set
LoRA adapter to load: /content/drive/MyDrive/semeval_august27neil_results/lora_best_dev


In [5]:
fact_checks     = pd.read_csv(os.path.join(TRAIN_DEV_DIR, 'fact_checks.csv'))
posts           = pd.read_csv(os.path.join(TRAIN_DEV_DIR, 'posts.csv'))
pairs_train_all = pd.read_csv(os.path.join(TRAIN_DEV_DIR, 'pairs.csv'))
pairs_dev_mono  = pd.read_csv(os.path.join(TRAIN_DEV_DIR, 'pairs_dev_monolingual.csv'))

with open(os.path.join(TRAIN_DEV_DIR, 'tasks.json'), 'r', encoding='utf-8') as f:
    tasks = json.load(f)

LANG = 'eng'
print(f'fact_checks: {fact_checks.shape}  posts: {posts.shape}  pairs: {pairs_train_all.shape}')

fact_checks: (153743, 4)  posts: (24431, 5)  pairs: (25743, 2)


In [6]:
eng_task = tasks['monolingual'][LANG]
eng_fact_check_ids  = set(eng_task['fact_checks'])
eng_posts_train_ids = set(eng_task['posts_train'])
eng_posts_dev_ids   = set(eng_task['posts_dev'])

eng_train_pairs = pairs_train_all[
    pairs_train_all['post_id'].isin(eng_posts_train_ids) &
    pairs_train_all['fact_check_id'].isin(eng_fact_check_ids)
].drop_duplicates().reset_index(drop=True)

eng_dev_pairs = pairs_dev_mono[
    pairs_dev_mono['post_id'].isin(eng_posts_dev_ids) &
    pairs_dev_mono['fact_check_id'].isin(eng_fact_check_ids)
].drop_duplicates().reset_index(drop=True)

print(f'eng-eng TRAIN pairs: {len(eng_train_pairs)}   eng-eng DEV pairs: {len(eng_dev_pairs)}')

eng-eng TRAIN pairs: 5446   eng-eng DEV pairs: 627


In [25]:
def safe_parse_tuple(s):
    if pd.isna(s):
        return None
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return None

def extract_text(parsed, fallback=''):
    if parsed is None or len(parsed) < 2:
        return fallback
    text = parsed[1] if parsed[1] else parsed[0]
    return text if isinstance(text, str) else fallback

def get_ocr_fallback(ocr_str):
    parsed = safe_parse_tuple(ocr_str)
    if isinstance(parsed, list) and len(parsed) > 0:
        return extract_text(parsed[0])
    return ''

def build_fused_text(text_field_str, ocr_field_str):
    main_text = extract_text(safe_parse_tuple(text_field_str)).strip()
    ocr_text  = get_ocr_fallback(ocr_field_str).strip()
    if not main_text and not ocr_text:
        return ''
    if not ocr_text:
        return main_text
    if not main_text:
        return ocr_text
    if ocr_text.lower() in main_text.lower():
        return main_text
    return main_text + ' ' + ocr_text

eng_claims_pool_df = fact_checks[fact_checks['fact_check_id'].isin(eng_fact_check_ids)].copy()
eng_claims_pool_df['claim_text'] = eng_claims_pool_df['claim'].apply(lambda s: extract_text(safe_parse_tuple(s)))
claim_id_to_text = dict(zip(eng_claims_pool_df['fact_check_id'], eng_claims_pool_df['claim_text']))
claim_id_to_text = {k: v for k, v in claim_id_to_text.items() if v.strip()}

eng_posts_dev_df = posts[posts['post_id'].isin(eng_posts_dev_ids)].copy()
eng_posts_dev_df['text_clean'] = eng_posts_dev_df.apply(lambda r: build_fused_text(r['text'], r['ocr']), axis=1)
post_id_to_text_dev = dict(zip(eng_posts_dev_df['post_id'], eng_posts_dev_df['text_clean']))
post_id_to_text_dev = {k: v for k, v in post_id_to_text_dev.items() if v.strip()}

# --- FIX: normalize claim keys to string, matching how downstream
# candidate lists store IDs (str(claim_id_list[j])) ---
claim_id_to_text = {str(k): v for k, v in claim_id_to_text.items()}
claim_ids = list(claim_id_to_text.keys())

print(f'Claim pool: {len(claim_id_to_text)}   Dev posts usable: {len(post_id_to_text_dev)}')
print(f'Sample claim_id_to_text key type: {type(next(iter(claim_id_to_text)))}')

Claim pool: 85734   Dev posts usable: 478
Sample claim_id_to_text key type: <class 'str'>


In [26]:
test_fact_checks = pd.read_csv(os.path.join(TEST_SET_DIR, 'fact_checks.csv'))
test_posts       = pd.read_csv(os.path.join(TEST_SET_DIR, 'posts.csv'))

with open(os.path.join(TEST_SET_DIR, 'tasks.json'), 'r', encoding='utf-8') as f:
    test_tasks = json.load(f)
with open(os.path.join(TEST_SET_DIR, 'monolingual_reference.json'), 'r', encoding='utf-8') as f:
    monolingual_reference = json.load(f)

eng_test_task = test_tasks['monolingual'][LANG]
eng_test_fact_check_ids = set(eng_test_task['fact_checks'])
eng_test_posts_ids      = set(eng_test_task['posts_test'])

eng_test_claims_pool_df = test_fact_checks[test_fact_checks['fact_check_id'].isin(eng_test_fact_check_ids)].copy()
eng_test_claims_pool_df['claim_text'] = eng_test_claims_pool_df['claim'].apply(lambda s: extract_text(safe_parse_tuple(s)))
test_claim_id_to_text = dict(zip(eng_test_claims_pool_df['fact_check_id'], eng_test_claims_pool_df['claim_text']))
test_claim_id_to_text = {k: v for k, v in test_claim_id_to_text.items() if v.strip()}

eng_test_posts_df = test_posts[test_posts['post_id'].isin(eng_test_posts_ids)].copy()
eng_test_posts_df['text_clean'] = eng_test_posts_df.apply(lambda r: build_fused_text(r['text'], r['ocr']), axis=1)
test_post_id_to_text = dict(zip(eng_test_posts_df['post_id'], eng_test_posts_df['text_clean']))
test_post_id_to_text = {k: v for k, v in test_post_id_to_text.items() if v.strip()}

# --- FIX: normalize claim keys to string, matching how downstream
# candidate lists store IDs (str(claim_id_list[j])) ---
test_claim_id_to_text = {str(k): v for k, v in test_claim_id_to_text.items()}
test_claim_ids       = list(test_claim_id_to_text.keys())
test_claim_texts     = [test_claim_id_to_text[c] for c in test_claim_ids]
test_post_ids_list   = [pid for pid in eng_test_posts_ids if pid in test_post_id_to_text]
test_post_texts_list = [test_post_id_to_text[pid] for pid in test_post_ids_list]

eng_test_reference = {
    str(pid): [str(fc) for fc in fcs]
    for pid, fcs in monolingual_reference.items()
    if int(pid) in eng_test_posts_ids
}

print(f'TEST claim pool: {len(test_claim_ids)}   TEST posts: {len(test_post_ids_list)}   GT: {len(eng_test_reference)}')
print(f'Sample test_claim_id_to_text key type: {type(next(iter(test_claim_id_to_text)))}')

TEST claim pool: 145287   TEST posts: 500   GT: 500
Sample test_claim_id_to_text key type: <class 'str'>


In [27]:
def success_at_k(retrieved_ranked, ground_truth, k):
    hits = 0
    for pid, true_fcs in ground_truth.items():
        if pid not in retrieved_ranked:
            continue
        if any(fc in true_fcs for fc in retrieved_ranked[pid][:k]):
            hits += 1
    return hits / len(ground_truth)

def evaluate_all_k(retrieved_ranked, ground_truth, ks=(1, 5, 10)):
    results = {}
    for k in ks:
        score = success_at_k(retrieved_ranked, ground_truth, k)
        results[f'success@{k}'] = score
        print(f'success@{k}: {score:.4f}')
    return results

def mrr(retrieved_ranked, ground_truth, k=10):
    scores = []
    for pid, true_fcs in ground_truth.items():
        if pid not in retrieved_ranked:
            scores.append(0.0); continue
        rr = 0.0
        for rank, cid in enumerate(retrieved_ranked[pid][:k], start=1):
            if cid in true_fcs:
                rr = 1.0 / rank; break
        scores.append(rr)
    return float(np.mean(scores))

def ndcg_at_k(retrieved_ranked, ground_truth, k=10):
    scores = []
    for pid, true_fcs in ground_truth.items():
        if pid not in retrieved_ranked:
            scores.append(0.0); continue
        dcg = 0.0
        for rank, cid in enumerate(retrieved_ranked[pid][:k], start=1):
            if cid in true_fcs:
                dcg += 1.0 / math.log2(rank + 1)
        n_relevant = min(len(true_fcs), k)
        idcg = sum(1.0 / math.log2(r + 1) for r in range(1, n_relevant + 1))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))

In [28]:
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

MODEL_NAME = 'intfloat/multilingual-e5-large'
MAX_LEN = 128
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

print('Loading frozen base ME5-Large...')
frozen_model = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float16).to(device)
frozen_model.eval()

print('Loading fine-tuned ME5-Large (LoRA adapter)...')
eval_base_model = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float16).to(device)
eval_model = PeftModel.from_pretrained(eval_base_model, FINAL_ADAPTER_PATH)
eval_model.eval()

print('Both models loaded — nothing retrained.')

Loading frozen base ME5-Large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading fine-tuned ME5-Large (LoRA adapter)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Both models loaded — nothing retrained.


In [29]:
@torch.no_grad()
def encode_texts_eval(texts, prefix, model, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f'Encoding ({prefix.strip(": ")})', leave=False):
        batch = [prefix + t for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(**enc)
            emb = mean_pool(out.last_hidden_state, enc['attention_mask'])
            emb = F.normalize(emb, p=2, dim=1)
        all_embs.append(emb.cpu())
        del enc, out, emb
    return torch.cat(all_embs, dim=0)

In [30]:
print('Encoding TEST — fine-tuned model...')
test_claim_embs = encode_texts_eval(test_claim_texts, prefix='passage: ', model=eval_model)
test_post_embs  = encode_texts_eval(test_post_texts_list, prefix='query: ', model=eval_model)

print('Encoding TEST — frozen model...')
frozen_claim_embs = encode_texts_eval(test_claim_texts, prefix='passage: ', model=frozen_model)
frozen_post_embs  = encode_texts_eval(test_post_texts_list, prefix='query: ', model=frozen_model)

torch.cuda.empty_cache()

def rank_topk(post_embs, claim_embs, k, ids_post, ids_claim):
    claim_gpu = claim_embs.to(device).float()
    ranked = {}
    for start in range(0, len(ids_post), 128):
        chunk = post_embs[start:start+128].to(device).float()
        sims = chunk @ claim_gpu.T
        topk_idx = torch.topk(sims, k=k, dim=1).indices
        for i, pid in enumerate(ids_post[start:start+128]):
            ranked[str(pid)] = [str(ids_claim[j]) for j in topk_idx[i].cpu().tolist()]
        del chunk, sims, topk_idx
    del claim_gpu
    torch.cuda.empty_cache()
    return ranked

retrieved_finetuned = rank_topk(test_post_embs, test_claim_embs, 10, test_post_ids_list, test_claim_ids)
retrieved_frozen     = rank_topk(frozen_post_embs, frozen_claim_embs, 10, test_post_ids_list, test_claim_ids)

print('\n=== Sanity: fine-tuned only ===')
results_finetuned = evaluate_all_k(retrieved_finetuned, eng_test_reference)
print('\n=== Sanity: frozen only ===')
results_frozen_fused = evaluate_all_k(retrieved_frozen, eng_test_reference)

Encoding TEST — fine-tuned model...


Encoding TEST — frozen model...



=== Sanity: fine-tuned only ===
success@1: 0.4180
success@5: 0.7320
success@10: 0.8120

=== Sanity: frozen only ===
success@1: 0.4320
success@5: 0.7180
success@10: 0.7780


In [31]:
dev_post_ids_list = [pid for pid in eng_dev_pairs['post_id'].unique() if pid in post_id_to_text_dev]
dev_post_texts_list = [post_id_to_text_dev[pid] for pid in dev_post_ids_list]
dev_truth = {}
for _, row in eng_dev_pairs.iterrows():
    if row['post_id'] in dev_post_ids_list:
        dev_truth.setdefault(str(row['post_id']), []).append(str(row['fact_check_id']))

claim_texts_pool = [claim_id_to_text[cid] for cid in claim_ids]

print('Encoding DEV — fine-tuned model...')
dev_claim_embs_ft = encode_texts_eval(claim_texts_pool, prefix='passage: ', model=eval_model)
dev_post_embs_ft  = encode_texts_eval(dev_post_texts_list, prefix='query: ', model=eval_model)

print('Encoding DEV — frozen model...')
dev_claim_embs_fr = encode_texts_eval(claim_texts_pool, prefix='passage: ', model=frozen_model)
dev_post_embs_fr  = encode_texts_eval(dev_post_texts_list, prefix='query: ', model=frozen_model)

dev_sims_ft = dev_post_embs_ft.float() @ dev_claim_embs_ft.float().T
dev_sims_fr = dev_post_embs_fr.float() @ dev_claim_embs_fr.float().T

best_alpha, best_score = 0.5, -1
for alpha in np.arange(0.0, 1.01, 0.05):
    combo = alpha * dev_sims_ft + (1 - alpha) * dev_sims_fr
    topk_idx = torch.topk(combo, k=10, dim=1).indices
    dev_ranked = {str(pid): [str(claim_ids[j]) for j in topk_idx[i].tolist()]
                  for i, pid in enumerate(dev_post_ids_list)}
    score = success_at_k(dev_ranked, dev_truth, 10)
    if score > best_score:
        best_score, best_alpha = score, alpha

print(f'\nBest alpha on DEV: {best_alpha:.2f}  (dev success@10 = {best_score:.4f})')

Encoding DEV — fine-tuned model...


Encoding DEV — frozen model...



Best alpha on DEV: 0.20  (dev success@10 = 0.8452)


In [32]:
test_sims_ft = test_post_embs.float() @ test_claim_embs.float().T
test_sims_fr = frozen_post_embs.float() @ frozen_claim_embs.float().T
combo_test = best_alpha * test_sims_ft + (1 - best_alpha) * test_sims_fr

topk_idx = torch.topk(combo_test, k=10, dim=1).indices
retrieved_ranked_ensemble = {
    str(pid): [str(test_claim_ids[j]) for j in topk_idx[i].tolist()]
    for i, pid in enumerate(test_post_ids_list)
}

print(f'=== Ensemble (alpha={best_alpha:.2f}) — TEST baseline to beat ===')
results_ensemble = evaluate_all_k(retrieved_ranked_ensemble, eng_test_reference)

=== Ensemble (alpha=0.20) — TEST baseline to beat ===
success@1: 0.4760
success@5: 0.7660
success@10: 0.8340


In [33]:
combo_dev = best_alpha * dev_sims_ft + (1 - best_alpha) * dev_sims_fr
topk50_idx = torch.topk(combo_dev, k=50, dim=1).indices
dev_ranked_top50 = {str(pid): [str(claim_ids[j]) for j in topk50_idx[i].tolist()]
                     for i, pid in enumerate(dev_post_ids_list)}

dev_s10 = success_at_k(dev_ranked_top50, dev_truth, 10)
dev_s50 = success_at_k(dev_ranked_top50, dev_truth, 50)
print(f'DEV success@10: {dev_s10:.4f}   DEV success@50: {dev_s50:.4f}   Headroom: {dev_s50-dev_s10:+.4f}')
print('✅ Headroom exists' if dev_s50 - dev_s10 >= 0.02 else '⚠️ Little headroom — proceed cautiously')

DEV success@10: 0.8452   DEV success@50: 0.9059   Headroom: +0.0607
✅ Headroom exists


In [34]:
def build_stage1_candidates_biencoder_only(post_ids, ensemble_topk_idx, claim_id_list):
    return {
        str(pid): [str(claim_id_list[j]) for j in ensemble_topk_idx[i].tolist()]
        for i, pid in enumerate(post_ids)
    }

dev_candidates = build_stage1_candidates_biencoder_only(dev_post_ids_list, topk50_idx, claim_ids)

test_topk50_idx = torch.topk(combo_test, k=50, dim=1).indices
test_candidates = build_stage1_candidates_biencoder_only(test_post_ids_list, test_topk50_idx, test_claim_ids)

print(f'DEV candidates built for {len(dev_candidates)} posts, top-50 each (bi-encoder only, no BM25)')
print(f'DEV recall@50: {success_at_k(dev_candidates, dev_truth, 50):.4f}')

DEV candidates built for 478 posts, top-50 each (bi-encoder only, no BM25)
DEV recall@50: 0.9059


In [35]:
del eval_model, eval_base_model, frozen_model
gc.collect()
torch.cuda.empty_cache()

print(f'GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'GPU reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB')

GPU allocated: 1.15 GB
GPU reserved : 1.16 GB


In [36]:
from sentence_transformers import CrossEncoder

torch.cuda.empty_cache()
try:
    reranker = CrossEncoder(
        'BAAI/bge-reranker-v2-m3',
        max_length=256,
        device=device,
        automodel_args={'torch_dtype': torch.float16},
    )
    print('Reranker loaded on GPU (fp16).')
except torch.cuda.OutOfMemoryError:
    print('OOM on GPU — falling back to CPU (slower but safe).')
    reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=256, device='cpu')

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker loaded on GPU (fp16).


In [44]:
def rerank_pipeline(post_ids, post_texts, candidates_dict, claim_text_lookup, beta=0.5, top_k=10):
    """beta weights the cross-encoder score vs the original retriever rank position.
       beta=1.0 -> pure cross-encoder ranking. beta=0.0 -> pure retriever order."""
    final_ranked = {}
    for pid, post_text in zip(post_ids, post_texts):
        cand_ids = candidates_dict[str(pid)]
        valid_ids = [cid for cid in cand_ids if cid in claim_text_lookup]
        if not valid_ids:
            final_ranked[str(pid)] = cand_ids[:top_k]
            continue
        pairs = [(post_text, claim_text_lookup[cid]) for cid in valid_ids]
        ce_scores = np.array(reranker.predict(pairs, batch_size=32, show_progress_bar=False))
        ce_norm = (ce_scores - ce_scores.min()) / (ce_scores.max() - ce_scores.min() + 1e-9)
        retriever_score = np.linspace(1.0, 0.0, num=len(valid_ids))  # rank-position proxy, best=1
        fused = beta * ce_norm + (1 - beta) * retriever_score
        order = np.argsort(-fused)
        final_ranked[str(pid)] = [valid_ids[i] for i in order[:top_k]]
    return final_ranked

In [45]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import random

random.seed(42)

# ---- 1. Build train-post text lookup (wasn't built earlier — only dev/test were) ----
eng_posts_train_df = posts[posts['post_id'].isin(eng_posts_train_ids)].copy()
eng_posts_train_df['text_clean'] = eng_posts_train_df.apply(lambda r: build_fused_text(r['text'], r['ocr']), axis=1)
post_id_to_text_train = dict(zip(eng_posts_train_df['post_id'], eng_posts_train_df['text_clean']))
post_id_to_text_train = {k: v for k, v in post_id_to_text_train.items() if v.strip()}

# ---- 2. Build (post, claim, label) examples: positives from ground truth, negatives sampled from the claim pool ----
N_NEG_PER_POS = 4
train_examples = []
all_claim_ids_set = set(claim_id_to_text.keys())
grouped = eng_train_pairs.groupby('post_id')['fact_check_id'].apply(list).to_dict()

for pid, true_fcs in grouped.items():
    if pid not in post_id_to_text_train:
        continue
    post_text = post_id_to_text_train[pid]
    true_fcs_str = [str(fc) for fc in true_fcs]

    for fc in true_fcs_str:
        if fc not in claim_id_to_text:
            continue
        train_examples.append(InputExample(texts=[post_text, claim_id_to_text[fc]], label=1.0))
        neg_pool = list(all_claim_ids_set - set(true_fcs_str))
        for neg in random.sample(neg_pool, min(N_NEG_PER_POS, len(neg_pool))):
            train_examples.append(InputExample(texts=[post_text, claim_id_to_text[neg]], label=0.0))

random.shuffle(train_examples)
print(f'Built {len(train_examples)} CE training examples '
      f'({sum(e.label==1.0 for e in train_examples)} pos / {sum(e.label==0.0 for e in train_examples)} neg)')

# ---- 3. Fine-tune the cross-encoder ----
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

reranker_ft = CrossEncoder(
    'BAAI/bge-reranker-v2-m3',
    max_length=256,
    device=device,
    automodel_args={'torch_dtype': torch.float32},  # fp32 for stable training
)

reranker_ft.fit(
    train_dataloader=train_dataloader,
    epochs=1,
    warmup_steps=int(0.1 * len(train_dataloader)),
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True,
)

reranker = reranker_ft  # rerank_pipeline() uses the global `reranker`, so this swaps it in
print('Fine-tuned reranker swapped in.')

# ---- 4. Re-sweep beta on DEV with the fine-tuned CE ----
best_beta, best_beta_score = 0.5, -1
for beta in np.arange(0.0, 1.01, 0.1):
    ranked = rerank_pipeline(dev_post_ids_list, dev_post_texts_list, dev_candidates, claim_id_to_text, beta=beta)
    score = success_at_k(ranked, dev_truth, 10)
    print(f'beta={beta:.1f}  dev success@10={score:.4f}')
    if score > best_beta_score:
        best_beta_score, best_beta = score, beta
print(f'\nBest beta on DEV (fine-tuned CE): {best_beta:.2f}  (success@10={best_beta_score:.4f})')

# ---- 5. Apply to TEST and compare ----
FINAL_BETA = best_beta
test_reranked_ft = rerank_pipeline(test_post_ids_list, test_post_texts_list, test_candidates, test_claim_id_to_text, beta=FINAL_BETA)

print('\n=== TEST: ensemble baseline (no rerank) ===')
results_ensemble = evaluate_all_k(retrieved_ranked_ensemble, eng_test_reference)

print(f'\n=== TEST: ensemble + FINE-TUNED cross-encoder rerank (beta={FINAL_BETA:.2f}) ===')
results_reranked_ft = evaluate_all_k(test_reranked_ft, eng_test_reference)

Built 27230 CE training examples (5446 pos / 21784 neg)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Step,Training Loss
500,0.102402
1000,0.063026
1500,0.059209


Fine-tuned reranker swapped in.
beta=0.0  dev success@10=0.8452
beta=0.1  dev success@10=0.8473
beta=0.2  dev success@10=0.8473
beta=0.3  dev success@10=0.8452
beta=0.4  dev success@10=0.8431
beta=0.5  dev success@10=0.8452
beta=0.6  dev success@10=0.8452
beta=0.7  dev success@10=0.8473
beta=0.8  dev success@10=0.8473
beta=0.9  dev success@10=0.8494
beta=1.0  dev success@10=0.8577

Best beta on DEV (fine-tuned CE): 1.00  (success@10=0.8577)

=== TEST: ensemble baseline (no rerank) ===
success@1: 0.4760
success@5: 0.7660
success@10: 0.8340

=== TEST: ensemble + FINE-TUNED cross-encoder rerank (beta=1.00) ===
success@1: 0.4400
success@5: 0.7820
success@10: 0.8600


In [3]:
print(os.path.exists(os.path.join(OUTPUT_DIR, 'reranker_finetuned_v1')))
print(os.path.exists(os.path.join(OUTPUT_DIR, 'reranker_finetuned_minilm_v1')))

NameError: name 'OUTPUT_DIR' is not defined

In [2]:
import os
out_dir = os.path.join(OUTPUT_DIR, 'reranking_pipeline_results_CE_FINETUNED')
os.makedirs(out_dir, exist_ok=True)

comparison = pd.DataFrame({
    'Frozen ME5-Large': results_frozen_fused,
    'Fine-tuned ME5-Large (LoRA)': results_finetuned,
    'Ensemble (dev-tuned alpha)': results_ensemble,
    'Ensemble + CE Rerank (fine-tuned)': results_reranked_ft,
}).T[['success@1', 'success@5', 'success@10']]

print(comparison.round(4).to_string())
comparison.to_csv(os.path.join(out_dir, 'comparison_ce_finetuned.csv'))
with open(os.path.join(out_dir, 'predictions_ce_finetuned_test.json'), 'w') as f:
    json.dump(test_reranked_ft, f, indent=2)
print(f'\nSaved to: {out_dir}')

NameError: name 'OUTPUT_DIR' is not defined

In [43]:
n_top1_changed = 0
n_top10_set_changed = 0
all_ce_stds = []

for pid in test_post_ids_list[:200]:
    pid_str = str(pid)
    baseline_top10 = set(retrieved_ranked_ensemble.get(pid_str, []))
    reranked_top10 = set(test_reranked_ft.get(pid_str, []))

    baseline_top1 = retrieved_ranked_ensemble.get(pid_str, [None])[0]
    reranked_top1 = test_reranked_ft.get(pid_str, [None])[0]

    if baseline_top1 != reranked_top1:
        n_top1_changed += 1
    if baseline_top10 != reranked_top10:
        n_top10_set_changed += 1

    cand_ids = test_candidates[pid_str]
    valid_ids = [c for c in cand_ids if c in test_claim_id_to_text]
    post_text = test_post_id_to_text[pid]
    if valid_ids:
        pairs = [(post_text, test_claim_id_to_text[c]) for c in valid_ids]
        scores = reranker.predict(pairs, show_progress_bar=False)
        all_ce_stds.append(np.std(scores))

print(f'Posts where TOP-1 changed        : {n_top1_changed} / 200')
print(f'Posts where TOP-10 SET changed    : {n_top10_set_changed} / 200')
print(f'Mean CE score std across posts    : {np.mean(all_ce_stds):.6f}')
print(f'Min / Max CE score std            : {np.min(all_ce_stds):.6f} / {np.max(all_ce_stds):.6f}')

Posts where TOP-1 changed        : 0 / 200
Posts where TOP-10 SET changed    : 0 / 200
Mean CE score std across posts    : 0.186806
Min / Max CE score std            : 0.000011 / 0.432537
